# 02 ニューラルネットによる相転移の検出（Python / Keras）

新潟大学 集中講義「機械学習と物理学」（2026年9月）の計算実習です。講義ノートの第4・5章に対応します。

イジング模型の配位を画像として分類器に入れ、**低温と高温の配位だけ**で「秩序相か無秩序相か」を学習させます。
そのあと全温度の配位に適用し、秩序相と判定される確率 $P_{\rm ord}(T)$ から転移温度を読みます。
最後に、分類器が配位の何を見ていたのかを、磁化 $|m|$ やエネルギー $e$ と比べて調べます。

**Colab での準備**：「ランタイム」→「ランタイムのタイプを変更」で Python 3 を選びます（GPU があると学習が速くなりますが、なくても動きます）。
Keras は Colab に最初から入っています。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers, ops

keras.utils.set_random_seed(0)
TC = 2 / np.log(1 + np.sqrt(2))   # 厳密な臨界温度 2.269...
print("keras", keras.__version__)

## 1. データを読み込む

ノートブック 01（Julia）で作った `ising_L16.npz` を使います。
- 自分で作ったファイルを使う場合は、`USE_UPLOAD = True` にしてセルを実行し、ファイルを選んでアップロードします。
- そうでなければ、GitHub に置いた同じ形式のデータを読み込みます。

中身は、配位 `configs`（形は (配位数, L, L)、値は ±1）、温度 `T`、連鎖の番号 `chain`（各温度で 0〜3 の4本）です。

In [ ]:
USE_UPLOAD = False

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    fname = list(uploaded)[0]
else:
    import urllib.request
    fname = "ising_L16.npz"
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/akio-tomiya/niigata2026-ml-physics/main/data/ising_L16.npz", fname)

d = np.load(fname)
X = d["configs"].astype("float32")
T = d["T"]
chain = d["chain"]
L = int(d["L"])
temps = np.unique(T)
print(X.shape, "temperatures:", temps.min(), "...", temps.max(), " L =", L)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, t in zip(axes, [1.5, 2.3, 3.5]):
    k = np.where(np.isclose(T, t))[0][0]
    ax.imshow(X[k], cmap="cividis", vmin=-1, vmax=1)
    ax.set_title(f"T = {t}"); ax.axis("off")
plt.show()

## 2. 物理量で見る：磁化とエネルギー

配位ごとに磁化 $m=\frac1N\sum_i s_i$ とエネルギー密度 $e=H/N$ を計算し、温度ごとに平均します（講義ノート第1章・第3章）。

In [ ]:
m = X.mean(axis=(1, 2))
e = -(X * np.roll(X, 1, axis=1) + X * np.roll(X, 1, axis=2)).sum(axis=(1, 2)) / L**2
absm_T = [np.abs(m[T == t]).mean() for t in temps]
e_T = [e[T == t].mean() for t in temps]

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot(temps, absm_T, "o-"); axes[0].set_ylabel("<|m|>")
axes[1].plot(temps, e_T, "o-"); axes[1].set_ylabel("<e>")
for ax in axes:
    ax.axvline(TC, ls="--", color="gray"); ax.set_xlabel("T")
plt.tight_layout(); plt.show()

## 3. ラベルとデータ分割

- ラベル：$T\le 2.0$ の配位を秩序相 $y=1$、$T\ge 2.6$ の配位を無秩序相 $y=0$ とします。$2.0<T<2.6$ の配位は**学習に使いません**。
- 分割：同じ連鎖から取った配位はよく似ているので、**連鎖ごと**に分けます（講義ノート第5章「データ分割」）。
  連鎖 0, 1 を訓練、連鎖 2 を検証、連鎖 3 をテストに使います。

In [ ]:
labeled = (T <= 2.0) | (T >= 2.6)
y = (T <= 2.0).astype("float32")

train = labeled & (chain <= 1)
val = labeled & (chain == 2)
test = chain == 3          # テストは全温度で行う

# 全スピン反転 s -> -s でも相は変わらないので、訓練データに反転した配位も加える（データ拡張）
X_tr = np.concatenate([X[train], -X[train]]); y_tr = np.concatenate([y[train], y[train]])
X_va, y_va = X[val], y[val]
print("train", len(X_tr), " val", len(X_va), " test", test.sum())

## 4. 基準：$|m|$ だけを入力するロジスティック回帰

まず、物理量 $|m|$ 1つだけを入力とするロジスティック回帰 $q=\sigma(w|m|+b)$ を学習させます（講義ノート第4章）。
これは Keras では、入力1個・出力1個の全結合層にシグモイド関数を付けたものです。

In [ ]:
absm = np.abs(m)[:, None]
logreg = keras.Sequential([keras.Input(shape=(1,)), layers.Dense(1, activation="sigmoid")])
logreg.compile(optimizer=keras.optimizers.Adam(0.05), loss="binary_crossentropy", metrics=["accuracy"])
logreg.fit(absm[train], y[train], epochs=100, batch_size=64, verbose=0,
           validation_data=(absm[val], y[val]))
w, b = logreg.layers[0].get_weights()
print("w =", w.ravel(), " b =", b, " threshold |m|* =", -b / w.ravel())
print("validation accuracy:", logreg.evaluate(absm[val], y[val], verbose=0)[1])

## 5. 全結合ニューラルネット

配位 $s$（$L\times L$ の画像）をそのまま入力します。$L^2=256$ 個の入力 → 中間層64ユニット（ReLU）→ 出力1個（シグモイド）です（講義ノート第4章）。
損失は2値交差エントロピー、最適化は Adam です。検証データの損失が下がらなくなったら学習を止めます（早期終了）。

In [ ]:
def make_fnn():
    return keras.Sequential([
        keras.Input(shape=(L, L)),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])

fnn = make_fnn()
fnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
stop = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
hist = fnn.fit(X_tr, y_tr, epochs=50, batch_size=64, validation_data=(X_va, y_va),
               callbacks=[stop], verbose=0)
plt.plot(hist.history["loss"], label="train"); plt.plot(hist.history["val_loss"], label="validation")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.show()
print("validation accuracy:", fnn.evaluate(X_va, y_va, verbose=0)[1])

## 6. 畳み込みニューラルネット（CNN）

$3\times3$ のフィルター4枚で格子全体をなぞり、ReLU の後に格子全体で平均して、最後に1つの出力にします（講義ノート第4章）。
格子は周期境界なので、畳み込みの前に、反対側の端の値を貼り付ける「周期パディング」を自分で書きます。

In [ ]:
class PeriodicPad(layers.Layer):
    # 周期境界：上下左右に反対側の1行・1列を貼り付ける
    def call(self, x):
        x = ops.concatenate([x[:, -1:], x, x[:, :1]], axis=1)
        return ops.concatenate([x[:, :, -1:], x, x[:, :, :1]], axis=2)

def make_cnn():
    return keras.Sequential([
        keras.Input(shape=(L, L)),
        layers.Reshape((L, L, 1)),
        PeriodicPad(),
        layers.Conv2D(4, 3, activation="relu"),
        layers.GlobalAveragePooling2D(),
        layers.Dense(1, activation="sigmoid"),
    ])

cnn = make_cnn()
cnn.compile(optimizer=keras.optimizers.Adam(1e-2), loss="binary_crossentropy", metrics=["accuracy"])
cnn.fit(X_tr, y_tr, epochs=50, batch_size=64, validation_data=(X_va, y_va),
        callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)], verbose=0)
cnn.summary()
print("validation accuracy:", cnn.evaluate(X_va, y_va, verbose=0)[1])

## 7. 出力を温度の関数に戻す

テスト用の連鎖（学習に使っていない連鎖3）の配位を**全温度**で分類器に入れ、温度ごとに出力を平均します。
$$P_{\rm ord}(T)=\frac{1}{N_T}\sum_{k} q_\theta(y=1\mid s_k)$$
$P_{\rm ord}=1/2$ を横切る温度 $T^*$ を、隣り合う2点の線形補間で求めます（講義ノート第5章）。

In [ ]:
def p_ord(predict):
    return np.array([predict(t).mean() for t in temps])

def crossing(P):
    for i in range(len(temps) - 1):
        if P[i] >= 0.5 > P[i + 1]:
            return temps[i] + (P[i] - 0.5) / (P[i] - P[i + 1]) * (temps[i + 1] - temps[i])
    return np.nan

curves = {
    "|m| logistic": p_ord(lambda t: logreg.predict(absm[test & (T == t)], verbose=0)),
    "FNN": p_ord(lambda t: fnn.predict(X[test & (T == t)], verbose=0)),
    "CNN": p_ord(lambda t: cnn.predict(X[test & (T == t)], verbose=0)),
}
for name, P in curves.items():
    plt.plot(temps, P, "o-", label=f"{name}: T* = {crossing(P):.3f}")
plt.axvline(TC, ls="--", color="gray", label="exact T_c")
plt.axvspan(2.0, 2.6, color="gray", alpha=0.1)
plt.xlabel("T"); plt.ylabel("P_ord(T)"); plt.legend(); plt.show()

**問**：3つの $T^*$ を厳密な $T_c=2.269$ と比べてみましょう。灰色の範囲（$2.0<T<2.6$）は学習に使っていない温度です。
ニューラルネットの $T^*$ は、重みの初期値やミニバッチの選び方によって学習のたびに変わります。
最初のコードセルの `keras.utils.set_random_seed(0)` の値を変えて何回か学習し、$T^*$ のばらつきを見積もってみましょう。

## 8. 分類器は何を見ていたか

臨界付近（$2.0<T<2.6$）のテスト配位について、ニューラルネットの判定（$q>1/2$ なら秩序相）が、
- $|m|$ だけのしきい値による判定、
- エネルギー $e$ だけのしきい値による判定

とどれだけ一致するか（一致率）を調べます。しきい値は訓練データで正解率が最大になるように選びます（講義ノート第5章）。

In [ ]:
def best_threshold(x, yy, greater=True):
    cands = np.unique(x)
    accs = [((x > c) == yy).mean() if greater else ((x < c) == yy).mean() for c in cands]
    return cands[int(np.argmax(accs))]

m_star = best_threshold(np.abs(m[train]), y[train] == 1, greater=True)
e_star = best_threshold(e[train], y[train] == 1, greater=False)
crit = test & (T > 2.0) & (T < 2.6)
by_m = np.abs(m[crit]) > m_star
by_e = e[crit] < e_star
for name, model in [("FNN", fnn), ("CNN", cnn)]:
    by_nn = model.predict(X[crit], verbose=0).ravel() > 0.5
    print(f"{name}: agreement with |m| = {(by_nn == by_m).mean():.3f},  with e = {(by_nn == by_e).mean():.3f}")
print("thresholds: |m|* =", m_star, " e* =", e_star)

**問**：全結合ネットと CNN は、それぞれ $|m|$ と $e$ のどちらに近い判定をしていますか。
CNN は小さなフィルターで隣り合うスピンの組を見るので、どちらに近くなりそうかを考えてから結果を見てください。

## 9. 交差検証

検証データの正解率は、どの連鎖を検証に回したかで変わります。
4本の連鎖を順番に検証用にして4回学習し、正解率の平均とばらつきを見ます（$K$-分割交差検証、$K=4$）。

In [ ]:
accs = []
for k in range(4):
    tr_k = labeled & (chain != k)
    va_k = labeled & (chain == k)
    model = make_fnn()
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(np.concatenate([X[tr_k], -X[tr_k]]), np.concatenate([y[tr_k], y[tr_k]]),
              epochs=15, batch_size=64, verbose=0)
    accs.append(model.evaluate(X[va_k], y[va_k], verbose=0)[1])
print("accuracy per fold:", np.round(accs, 4))
print(f"mean = {np.mean(accs):.4f},  std = {np.std(accs):.4f}")

## 10. やってみよう（レポートの題材）

1. 訓練に使う温度を $T\le 1.8$ と $T\ge 3.0$ に変えると、$T^*$ はどう変わるか。$|m|$ の分類器と CNN で比べる。
2. ノートブック 01 で $L=8$ と $L=32$ のデータを作り、$T^*$ の $L$ 依存性を調べる（講義ノート第5章と付録H「有限サイズスケーリングによる外挿」）。
3. CNN のフィルターの重み `cnn.layers[2].get_weights()` を表示し、どのようなスピンの並びに反応するかを考える。
4. 入力の配位の各サイトを無作為に並べ替えると（`rng.permutation`）、全結合ネットと CNN の出力はそれぞれどう変わるか。